# Model C_5: label berita dengan LLM lokal (peristiwa, wilayah, arah dolar)

**Ide.** Model embedding (C_2, C_4) mengubah berita jadi angka yang sulit dibaca manusia. Di sini kita minta sebuah LLM kecil (Qwen2.5-3B-Instruct, jalan lokal di GPU) membaca SETIAP artikel dan menjawab tiga pertanyaan:
1. **event**: jenis peristiwa (kebijakan moneter, perang, tarif/perdagangan, minyak/energi, politik/pemilu, data ekonomi, pasar keuangan, lainnya);
2. **region**: wilayah utama (AS, Indonesia, Tiongkok, Timur Tengah, Eropa, lainnya);
3. **dollar**: apakah berita itu cenderung membuat dolar AS menguat (`stronger`), melemah (`weaker`), atau tidak jelas (`neutral`).

Lalu untuk tiap hari kita hitung **porsi** tiap label (mis. 30% artikel hari itu tentang minyak, 5 menguat vs 1 melemah). Hasilnya sekitar 20 kolom yang bisa dibaca manusia, dipakai LR dan HGB bersama fitur harga.

**Catatan jujur.**
- Tidak ada "jawaban benar" untuk label ini; LLM kecil bisa salah. Karena itu ada sel pemeriksaan manual (100 artikel) di bagian 5.
- Label "dollar" adalah pendapat model dan sebagian bergantung pada kata seperti "surge" atau "fall" (leak words yang memang kita pertahankan).
- Seperti C_2 dan C_4, ini hanya dinilai di validation. Test baru dipakai sekali untuk varian yang dipilih.

**Jalan penuh sekaligus.** Hasil tiap batch langsung ditulis ke `data/features_c/c5_llm_labels.jsonl`. Kalau terhenti (mati listrik, kehabisan memori), jalankan ulang: artikel yang sudah selesai dilewati. Perkiraan waktu: sekitar 1 sampai 2 jam di RTX 4060.

In [1]:
from c_common import *
import json, re
from pathlib import Path
import torch

D, text = load_all()   # corpus B_2: hanya artikel Inggris terfilter
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 5)


## 1. Pengaturan

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # sekitar 6 GB
CACHE_DIR = "D:/hf_cache"       # tempat model diunduh (drive D: karena C: hampir penuh). Folder dibuat otomatis.
BATCH = 16                      # kalau kehabisan memori GPU (CUDA out of memory), turunkan ke 8
MAX_WORDS = 110                 # tiap artikel dipotong ke 110 kata ("judul. lead")
MAX_NEW = 48                    # panjang jawaban maksimum (jawabannya pendek, satu baris JSON)
LIMIT = None                    # angka kecil (mis. 200) = uji cepat pada 200 artikel pertama. None = SEMUA artikel.

EVENTS  = ["monetary policy", "war/conflict", "trade/tariffs", "oil/energy", "elections/politics", "economic data", "financial markets", "other"]
REGIONS = ["US", "Indonesia", "China", "Middle East", "Europe", "other"]
DOLLAR  = ["stronger", "weaker", "neutral"]

f_out = FEATURES_C / "c5_llm_labels.jsonl"
TITLE, LEAD = "judul_gabungan", "lead_paragraph_gabungan"     # kolom corpus B_2 (sama dengan C_2)
FEATURES_C.mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

## 2. Satu baris per artikel: "judul. lead"

Sama dengan C_2 dan C_4: judul tiap artikel digabung dengan paragraf pembukanya. Judul dan lead tiap hari dipisah dengan `" | "`; kalau jumlahnya tidak cocok di suatu hari, hari itu hanya memakai judul.

In [3]:
def build_article_rows(text_df):
    """Satu baris per artikel. Return list (posisi_hari, id_artikel, teks)."""
    rows, art_id, fallback = [], 0, 0
    for pos, (t, l) in enumerate(zip(text_df[TITLE].fillna(""), text_df[LEAD].fillna(""))):
        titles = [x.strip() for x in t.split(" | ") if x.strip()]
        leads = [x.strip() for x in l.split(" | ")]
        if len(leads) != len(titles):          # jumlah tidak cocok -> judul saja
            leads, fallback = [""] * len(titles), fallback + 1
        for ti, li in zip(titles, leads):
            words = f"{ti}. {li}".strip(" .").split()
            rows.append((pos, art_id, " ".join(words[:MAX_WORDS])))
            art_id += 1
    print("hari memakai judul saja (jumlah judul/lead tidak cocok):", fallback)
    return rows


rows = build_article_rows(text)
has_en = (text["jumlah_berita"] > 0).values
if LIMIT:
    rows = rows[:LIMIT]
print(len(rows), "artikel | hari dengan artikel:", int(has_en.sum()))

hari memakai judul saja (jumlah judul/lead tidak cocok): 80
11764 artikel | hari dengan artikel: 1182


## 3. Muat LLM

Pertama kali dijalankan, model diunduh (sekitar 6 GB). Setelah itu dibaca dari cache. `padding_side="left"` diperlukan supaya jawaban semua artikel dalam satu batch mulai dari posisi yang sama.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left", cache_dir=CACHE_DIR)
llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, cache_dir=CACHE_DIR).eval()   # dimuat dulu di CPU, lalu dipindah ke GPU (tanpa paket accelerate)
llm.to("cuda" if torch.cuda.is_available() else "cpu")
print(MODEL_NAME, "| device:", llm.device, "| parameter:", round(sum(p.numel() for p in llm.parameters()) / 1e9, 1), "miliar")

W1115 15:16:26.894000 31432 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 434/434 [00:12<00:00, 33.43it/s]


Qwen/Qwen2.5-3B-Instruct | device: cuda:0 | parameter: 3.1 miliar


## 4. Pertanyaan ke LLM dan membaca jawabannya

Satu instruksi tetap untuk semua artikel. `do_sample=False` berarti jawaban selalu sama untuk teks yang sama (bisa diulang). Jawaban dibaca dengan `parse_answer`; kalau model menjawab di luar daftar, nilainya dianggap `other` / `neutral` dan dihitung sebagai "tidak valid".

In [5]:
SYSTEM = (
    "You classify short news items for a currency analyst. Answer with ONE line of JSON only, no other text:\n"
    '{"event": <one of ' + json.dumps(EVENTS) + '>, "region": <one of ' + json.dumps(REGIONS) + '>, '
    '"dollar": <one of ' + json.dumps(DOLLAR) + '>}\n'
    '"event" = the main type of event. "region" = the main country or region the news is about. '
    '"dollar" = whether this news would likely make the US dollar stronger, weaker, or neither (neutral).'
)

def make_prompt(article):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": article}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def _pick(raw, options, default):
    """Cocokkan jawaban ke salah satu pilihan (abaikan huruf besar/kecil). Tidak cocok -> default."""
    raw = (raw or "").strip().lower()
    for o in options:
        if raw == o.lower():
            return o, True
    for o in options:
        if o.lower() in raw:
            return o, True
    return default, False


def parse_answer(out):
    """Teks jawaban LLM -> (event, region, dollar, semua_valid)."""
    vals = {}
    for key in ("event", "region", "dollar"):
        m = re.search(rf'"{key}"\s*:\s*"([^"]*)"', out)
        vals[key] = m.group(1) if m else ""
    e, ok1 = _pick(vals["event"], EVENTS, "other")
    r, ok2 = _pick(vals["region"], REGIONS, "other")
    d, ok3 = _pick(vals["dollar"], DOLLAR, "neutral")
    return e, r, d, (ok1 and ok2 and ok3)


# uji kecil satu artikel (jawaban mentah), sebelum menjalankan semuanya
test_art = rows[0][2]
enc = tok([make_prompt(test_art)], return_tensors="pt").to(llm.device)
with torch.no_grad():
    o = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
print(test_art[:200], "\n->", tok.decode(o[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))

Weekly mortgage-refinance demand drops as interest rates stall. A prolonged period of low mortgage rates is taking its toll on the refinance market, as most borrowers who qualify have already gone thr 
-> {"event": "economic data", "region": "US", "dollar": "neutral"}


## 5. Jalankan untuk SEMUA artikel (bisa dilanjutkan kalau terhenti)

- Artikel diurutkan menurut panjang supaya padding minimal.
- Setiap batch langsung ditulis ke `c5_llm_labels.jsonl`. Saat dijalankan lagi, id yang sudah ada dilewati.
- Kalau muncul "CUDA out of memory", turunkan `BATCH` di sel pengaturan lalu jalankan ulang dari atas (progres tersimpan).

In [6]:


import time

done = {}
if f_out.exists():
    for line in f_out.read_text(encoding="utf-8").splitlines():
        if line.strip():
            r_ = json.loads(line)
            done[r_["id"]] = r_
todo = [r for r in rows if r[1] not in done]
print("sudah selesai:", len(done), "| tersisa:", len(todo))

todo.sort(key=lambda r: len(r[2]))
t0 = time.time()
with open(f_out, "a", encoding="utf-8") as fh:
    for s in range(0, len(todo), BATCH):
        batch = todo[s:s + BATCH]
        enc = tok([make_prompt(r[2]) for r in batch], return_tensors="pt", padding=True).to(llm.device)
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        for r, o in zip(batch, out):
            raw = tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            e, rg, d, ok = parse_answer(raw)
            rec = {"id": r[1], "pos": r[0], "event": e, "region": rg, "dollar": d, "valid": ok, "raw": raw}
            done[r[1]] = rec
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fh.flush()
        if (s // BATCH) % 25 == 0:
            el = time.time() - t0
            print(f"  {s + len(batch)}/{len(todo)} artikel | {el/60:.1f} menit | sisa sekitar {(len(todo)-s-len(batch)) * el / (s + len(batch)) / 60:.0f} menit")

labels = pd.DataFrame([done[r[1]] for r in rows])
print("selesai. artikel:", len(labels), "| jawaban valid:", round(labels["valid"].mean(), 3))

sudah selesai: 0 | tersisa: 11764
  16/11764 artikel | 0.1 menit | sisa sekitar 44 menit
  416/11764 artikel | 1.5 menit | sisa sekitar 40 menit
  816/11764 artikel | 2.9 menit | sisa sekitar 39 menit
  1216/11764 artikel | 4.4 menit | sisa sekitar 38 menit
  1616/11764 artikel | 5.8 menit | sisa sekitar 37 menit
  2016/11764 artikel | 9.3 menit | sisa sekitar 45 menit
  2416/11764 artikel | 13.3 menit | sisa sekitar 51 menit
  2816/11764 artikel | 17.3 menit | sisa sekitar 55 menit
  3216/11764 artikel | 21.3 menit | sisa sekitar 57 menit
  3616/11764 artikel | 25.3 menit | sisa sekitar 57 menit
  4016/11764 artikel | 29.5 menit | sisa sekitar 57 menit
  4416/11764 artikel | 33.6 menit | sisa sekitar 56 menit
  4816/11764 artikel | 38.3 menit | sisa sekitar 55 menit
  5216/11764 artikel | 43.0 menit | sisa sekitar 54 menit
  5616/11764 artikel | 47.6 menit | sisa sekitar 52 menit
  6016/11764 artikel | 52.2 menit | sisa sekitar 50 menit
  6416/11764 artikel | 56.7 menit | sisa sekitar

## 6. Periksa kualitas label

Dua pemeriksaan:
1. **Sebaran label**: kalau hampir semua artikel mendapat satu label yang sama (mis. 90% `neutral`), fiturnya tidak informatif.
2. **Cek manual**: sel di bawah menyimpan 100 artikel acak ke `c5_llm_check.csv`. Isi kolom `event_benar`, `region_benar`, `dollar_benar` sendiri (boleh hanya 50 baris pertama), lalu jalankan sel berikutnya untuk menghitung seberapa sering LLM setuju denganmu.

In [7]:
for c in ["event", "region", "dollar"]:
    print(labels[c].value_counts(normalize=True).round(3).to_string(), "\n")

art_text = {r[1]: r[2] for r in rows}
sample = labels.sample(min(100, len(labels)), random_state=SEED)[["id", "event", "region", "dollar"]].copy()
sample.insert(1, "teks", sample["id"].map(art_text))
for c in ["event_benar", "region_benar", "dollar_benar"]:
    sample[c] = ""
f_check = FEATURES_C / "c5_llm_check.csv"
if not f_check.exists():          # jangan timpa kalau kamu sudah mulai mengisi
    sample.to_csv(f_check, index=False, encoding="utf-8-sig")
    print("disimpan:", f_check)
sample.head(8)[["teks", "event", "region", "dollar"]]

event
monetary policy       0.318
economic data         0.248
financial markets     0.125
war/conflict          0.079
oil/energy            0.078
trade/tariffs         0.063
elections/politics    0.046
other                 0.043 

region
US             0.464
other          0.216
Middle East    0.151
Europe         0.093
China          0.073
Indonesia      0.002 

dollar
neutral     0.591
weaker      0.338
stronger    0.072 

disimpan: ..\data\features_c\c5_llm_check.csv


,teks,event,region,dollar
3717,"Fed holds off on rate hike, but says two more ...",monetary policy,US,neutral
11038,Saudi Arabia has ramped up oil shipments throu...,oil/energy,Middle East,neutral
2088,Chinese EV stocks tank after Li Auto and Xpeng...,financial markets,China,weaker
1399,Here are some strategies to pay off credit car...,monetary policy,US,weaker
1178,Biden waiving ethanol rule in bid to lower gas...,monetary policy,US,neutral
3116,"U.S. unveils $2 billion in Ukraine aid, new sa...",monetary policy,Europe,weaker
10673,Social Security COLA for 2027 may be higher as...,economic data,US,weaker
8045,Inflation readings and Treasury auctions to te...,economic data,US,neutral


In [8]:
chk = pd.read_csv(FEATURES_C / "c5_llm_check.csv")
for c in ["event", "region", "dollar"]:
    m = chk[f"{c}_benar"].notna() & (chk[f"{c}_benar"].astype(str).str.strip() != "")
    if m.sum():
        agree = (chk.loc[m, c].str.lower() == chk.loc[m, f"{c}_benar"].astype(str).str.strip().str.lower()).mean()
        print(f"{c}: setuju {agree:.0%} dari {int(m.sum())} artikel yang kamu isi")
    else:
        print(f"{c}: belum ada isian")

event: belum ada isian
region: belum ada isian
dollar: belum ada isian


## 7. Dari label per artikel ke fitur per hari

Untuk tiap hari: **porsi** artikel dengan tiap label (jumlah label dibagi jumlah artikel hari itu, supaya hari ramai tidak otomatis tampak "berbeda"), ditambah `dollar_net` = (porsi `stronger` dikurangi porsi `weaker`) dan penanda `has`. Hari tanpa artikel diisi 0 dengan `has = 0`. Tidak ada yang dipelajari dari validation/test di langkah ini, jadi tidak ada kebocoran.

In [9]:
def day_features(labels, n_days):
    cols = {}
    for kind, opts in [("event", EVENTS), ("region", REGIONS), ("dollar", DOLLAR)]:
        for o in opts:
            cols[f"c5_{kind}_{re.sub(r'[^a-z]+', '_', o.lower())}"] = (labels[kind] == o)
    ind = pd.DataFrame(cols)
    ind["pos"] = labels["pos"].values
    share = ind.groupby("pos").mean()
    out = pd.DataFrame(0.0, index=range(n_days), columns=share.columns)
    out.loc[share.index] = share.values
    out["c5_dollar_net"] = out["c5_dollar_stronger"] - out["c5_dollar_weaker"]
    return out


F = day_features(labels, len(text))
if LIMIT:      # uji cepat: hanya hari yang punya label
    has_en = has_en & np.isin(np.arange(len(text)), labels["pos"].values)
F["c5_has"] = has_en.astype(float)
F.index = text.index
LLM_COLS = list(F.columns)
for s in D:
    D[s] = D[s].drop(columns=[c for c in LLM_COLS if c in D[s].columns]).merge(F, left_on="date", right_index=True, how="left")
print(len(LLM_COLS), "kolom fitur:", LLM_COLS)

19 kolom fitur: ['c5_event_monetary_policy', 'c5_event_war_conflict', 'c5_event_trade_tariffs', 'c5_event_oil_energy', 'c5_event_elections_politics', 'c5_event_economic_data', 'c5_event_financial_markets', 'c5_event_other', 'c5_region_us', 'c5_region_indonesia', 'c5_region_china', 'c5_region_middle_east', 'c5_region_europe', 'c5_region_other', 'c5_dollar_stronger', 'c5_dollar_weaker', 'c5_dollar_neutral', 'c5_dollar_net', 'c5_has']


## 8. Nilai di validation

Dua model (LR dan HGB), harga + fitur label LLM. Skor utama: **AUC** (0,5 = sama dengan menebak acak). Laporkan jumlah konfigurasi: C_1: 2, C_2: 16, C_3: 4, C_4: 4, C_5: 2.

In [10]:
VARIANTS = {"C5_llm_LR": ("LR", LLM_COLS), "C5_llm_HGB": ("HGB", LLM_COLS)}
for name, (kind, cols) in VARIANTS.items():
    evaluate_val(name, kind, D, PRICE + cols, n_configs_tried=len(VARIANTS),
                 note=f"{MODEL_NAME}, label event/region/dollar per artikel -> porsi per hari ({len(cols)} kolom)")

C5_llm_LR [LR] val: DA=0.472, BalAcc=0.478, F1_macro=0.455, MCC=-0.047, AUC=0.509
C5_llm_HGB [HGB] val: DA=0.574, BalAcc=0.576, F1_macro=0.574, MCC=0.152, AUC=0.572


## 9. Bandingkan (validation)

- **harga saja**: model tanpa teks. Teks baru berguna kalau skornya jelas di atasnya.
- **C2_finbert_art_cls_HGB**: FinBERT terbaik di C_2.
- **C4_roberta_mean_***: RoBERTa dengan pooling utama.

Perbedaan AUC sekitar 0,04 atau kurang dengan 176 hari validation bisa muncul hanya karena kebetulan. Anggap satu model "lebih baik" hanya kalau selisihnya jelas lebih besar.

In [11]:
rows_cmp = []
for k in ["LR", "HGB"]:
    _, sc, _ = tune(k, D, PRICE)
    rows_cmp.append({"model": f"harga saja {k}", **sc})
for name in ["C2_finbert_art_cls_LR", "C2_finbert_art_cls_HGB", "C4_roberta_mean_LR", "C4_roberta_mean_HGB"] + list(VARIANTS):
    f = RESULTS_C / f"{name}.json"
    if f.exists():
        rows_cmp.append({"model": name, **json.loads(f.read_text())["val"]})
    else:
        print("belum ada:", name)
pd.DataFrame(rows_cmp).round(3)

,model,DA,BalAcc,F1_macro,MCC,AUC
0,harga saja LR,0.506,0.508,0.504,0.016,0.501
1,harga saja HGB,0.523,0.521,0.520,0.042,0.503
2,C2_finbert_art_cls_LR,0.506,0.505,0.505,0.010,0.486
3,C2_finbert_art_cls_HGB,0.591,0.588,0.588,0.179,0.568
4,C4_roberta_mean_LR,0.494,0.499,0.487,-0.002,0.488
5,C4_roberta_mean_HGB,0.562,0.564,0.562,0.128,0.596
6,C5_llm_LR,0.472,0.478,0.455,-0.047,0.509
7,C5_llm_HGB,0.574,0.576,0.574,0.152,0.572
